# Step 7 - Publication-Quality Visualisations

ROC / PR curves, confusion matrices, learning curves, calibration curves, native and permutation feature importance, and SHAP (beeswarm / summary / dependence) for the strongest models.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _find_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "data").is_dir():
            return cand
    return p

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as C
from src import viz
viz.setup_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Repository root:", ROOT)


Repository root: /Users/rauankaztaev/IdeaProjects/Draft/project


In [2]:
import joblib
from src import data, models, benchmark
from src import utils, feature_analysis as fa

df = data.load_clean()
x_train, x_test, y_train, y_test = data.get_splits(df)

# Reuse cached models/predictions from notebook 03 if available; else train.
store_path = C.MODELS_DIR / "test_predictions.joblib"
if store_path.exists():
    store = joblib.load(store_path)
    predictions = store["predictions"]
    fitted = {name: utils.load_model(name) for name in predictions}
    print("Loaded cached models & predictions.")
else:
    zoo = models.get_model_zoo(x_train)
    _, fitted, predictions = benchmark.evaluate_on_test(x_train, y_train, x_test, y_test, zoo)
    print("Trained models fresh.")
CURVE_MODELS = ["Random Forest", "XGBoost", "LightGBM", "CatBoost",
                "Logistic Regression", "SVM"]
CURVE_MODELS = [m for m in CURVE_MODELS if m in predictions]
BEST = "Random Forest" if "Random Forest" in fitted else CURVE_MODELS[0]
print("Curve models:", CURVE_MODELS, "| Best:", BEST)

Loaded cached models & predictions.
Curve models: ['Random Forest', 'XGBoost', 'LightGBM', 'CatBoost', 'Logistic Regression', 'SVM'] | Best: Random Forest


## 5.1 ROC and Precision-Recall curves

In [3]:
viz.plot_roc_curves(y_test.values, predictions, CURVE_MODELS)
viz.plot_pr_curves(y_test.values, predictions, CURVE_MODELS)
print("Saved roc_curves.png, pr_curves.png")

Saved roc_curves.png, pr_curves.png


## 5.2 Confusion matrices

In [4]:
viz.plot_confusion_grid(y_test.values, predictions, CURVE_MODELS)
viz.plot_confusion_matrix(y_test.values, predictions[BEST]["y_pred"], BEST,
                          fname="best_confusion_matrix.png")
print("Saved confusion_grid.png and best_confusion_matrix.png")

Saved confusion_grid.png and best_confusion_matrix.png


## 5.3 Calibration curves

In [5]:
viz.plot_calibration(y_test.values, predictions, CURVE_MODELS)
print("Saved calibration_curves.png")

Saved calibration_curves.png


## 5.4 Learning curves

In [6]:
for name in ["Random Forest", "XGBoost", "Logistic Regression"]:
    if name in fitted:
        viz.plot_learning_curve(fitted[name], x_train, y_train, name)
print("Saved learning curves.")

Saved learning curves.


## 5.5 Native feature importance (aggregated to original features)

In [7]:
for name in ["Random Forest", "Extra Trees", "XGBoost", "LightGBM", "CatBoost"]:
    if name in fitted:
        imp = fa.native_importance(fitted[name])
        if imp is not None:
            viz.plot_feature_importance(imp, name)
            utils.save_table(imp.reset_index().rename(columns={"index":"feature",0:"importance"}),
                             f"feature_importance_{name.lower().replace(' ','_')}")
print("Saved native feature-importance figures.")

Saved native feature-importance figures.


## 5.6 Permutation importance (model-agnostic, on the test set)

In [8]:
perm = fa.permutation_importance_df(fitted[BEST], x_test, y_test)
display(perm)
viz.plot_permutation_importance(perm, BEST)
utils.save_table(perm, "permutation_importance",
                 caption=f"Permutation importance ({BEST}).", label="tab:perm")

,feature,importance_mean,importance_std
0,Light,0.235934,0.009743
1,Fruit,0.196502,0.007814
2,Temp,0.127262,0.007752
3,Humidity,0.101529,0.007420
4,CO2,0.054923,0.005664


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/permutation_importance.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/permutation_importance.tex')}

## 5.7 SHAP analysis (beeswarm / summary / dependence)

In [9]:
import shap
shap_model = None
for cand in ["Random Forest", "XGBoost", "LightGBM", "CatBoost", "Extra Trees"]:
    if cand in fitted:
        shap_model = cand
        break
print("SHAP model:", shap_model)
sv, x_shap, names = fa.compute_shap(fitted[shap_model], x_train, x_test)

plt.figure()
shap.summary_plot(sv, x_shap, plot_type="bar", show=False)
plt.title(f"SHAP mean |value| - {shap_model}")
plt.tight_layout(); plt.savefig(C.FIGURES_DIR / "shap_summary_bar.png", dpi=300, bbox_inches="tight"); plt.close()

plt.figure()
shap.summary_plot(sv, x_shap, show=False)
plt.title(f"SHAP beeswarm - {shap_model}")
plt.tight_layout(); plt.savefig(C.FIGURES_DIR / "shap_beeswarm.png", dpi=300, bbox_inches="tight"); plt.close()

mean_abs = np.abs(sv).mean(0)
top_feat = names[int(np.argmax(mean_abs))]
plt.figure()
shap.dependence_plot(top_feat, sv, x_shap, show=False)
plt.title(f"SHAP dependence - {top_feat}")
plt.tight_layout(); plt.savefig(C.FIGURES_DIR / "shap_dependence.png", dpi=300, bbox_inches="tight"); plt.close()
print("Saved shap_summary_bar.png, shap_beeswarm.png, shap_dependence.png; top feature:", top_feat)

SHAP model: Random Forest


Saved shap_summary_bar.png, shap_beeswarm.png, shap_dependence.png; top feature: num__Light


**Observation.** SHAP, native importance, and permutation importance agree on the ordering of drivers, dominated by `Light`, `Humidity`, and `Temp`. The dependence plot shows the monotone effect of the top sensor on spoilage-risk log-odds.